# Module 04 — Agentic Loop

**Goal:** Turn the tool-caller from Module 03 into a real agent that recovers from errors, has a step budget, and remembers earlier turns.

## What's new vs Module 03

| Feature | Module 03 | Module 04 |
|---------|-----------|-----------|
| Tool calling | ✓ | ✓ |
| Error recovery (retry on bad SQL) | Partial | ✓ Full |
| Max-steps guard | ✗ | ✓ |
| Step-by-step reasoning log | Basic | Verbose |
| Multi-question memory | ✗ | ✓ |
| ReAct-style prompt | ✗ | ✓ |

Runs on the **Databricks free tier** against the built-in `samples` catalog — no data setup needed.

---
**Run each cell in order (Shift+Enter).**

## Step 1 — Install `openai`

In [ ]:
%pip install --quiet openai
dbutils.library.restartPython()

## Step 2 — Set your parameters

Pick any schema from the built-in `samples` catalog: `bakehouse`, `nyctaxi`, `accuweather`, `tpch`, etc. The agent introspects whatever you point it at.

In [ ]:
dbutils.widgets.text("API_KEY", "", "API key")
dbutils.widgets.text("BASE_URL", "https://models.github.ai/inference", "Base URL")
dbutils.widgets.text("MODEL", "openai/gpt-4o-mini", "Model")
dbutils.widgets.text("CATALOG", "samples", "Catalog")
dbutils.widgets.text("SCHEMA", "bakehouse", "Schema")

API_KEY = dbutils.widgets.get("API_KEY")
BASE_URL = dbutils.widgets.get("BASE_URL")
MODEL = dbutils.widgets.get("MODEL")
CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")

if not API_KEY:
    raise ValueError("Paste your API key into the API_KEY widget at the top of the notebook.")

from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

MAX_STEPS = 8
print(f"Model:    {MODEL}")
print(f"Database: {CATALOG}.{SCHEMA}")
print(f"MAX_STEPS: {MAX_STEPS}")

## Step 3 — Database and safety helpers

These are the building blocks the agent will call as tools.
- `get_schema()` — `SHOW TABLES` + `DESCRIBE TABLE` for every table in the schema
- `run_query(sql)` — `spark.sql(sql)` with a row cap
- `validate_sql(sql)` — guardrail that blocks anything that isn't a single SELECT

In [ ]:
import re, json
from dataclasses import dataclass, field

_FORBIDDEN = {"DROP","DELETE","UPDATE","INSERT","ALTER","TRUNCATE","CREATE","REPLACE","MERGE","EXEC","EXECUTE","GRANT","REVOKE","ATTACH","DETACH"}

@dataclass
class ValidationResult:
    is_safe: bool
    reason: str

def validate_sql(sql: str) -> ValidationResult:
    stripped = sql.strip()
    if not stripped:
        return ValidationResult(False, "SQL is empty.")
    cleaned = stripped.rstrip(";")
    if ";" in cleaned:
        return ValidationResult(False, "Only a single SELECT statement is allowed.")
    if cleaned.split()[0].upper() != "SELECT":
        return ValidationResult(False, f"Query must start with SELECT, got '{cleaned.split()[0]}'.")
    for kw in _FORBIDDEN:
        if re.search(rf"\b{kw}\b", cleaned.upper()):
            return ValidationResult(False, f"Forbidden keyword: {kw}")
    return ValidationResult(True, "ok")


def get_schema() -> dict:
    schema = {}
    tables = spark.sql("SHOW TABLES").collect()
    for row in tables:
        table_name = row.tableName
        cols = spark.sql(f"DESCRIBE TABLE {table_name}").collect()
        # DESCRIBE includes partition headers and blank rows — skip them
        schema[table_name] = [
            {"name": c.col_name, "type": c.data_type}
            for c in cols
            if c.col_name and not c.col_name.startswith("#")
        ]
    return schema


def run_query(sql: str):
    df = spark.sql(sql).limit(50)
    columns = df.columns
    rows = [row.asDict() for row in df.collect()]
    return columns, rows


_schema = get_schema()
print(f"Tables in {CATALOG}.{SCHEMA}: {list(_schema.keys())}")

## Step 4 — Tools and a ReAct-style system prompt

The tool definitions are the same as Module 03. The system prompt is new — it asks the model to **reason before acting** and tells it that errors will be returned so it can self-correct.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Inspect the database \u2014 returns all table names and column definitions.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_query",
            "description": (
                "Execute a read-only SELECT query. Returns rows as JSON. "
                "If the query fails or is blocked by the safety check, the error "
                "message is returned so you can fix the SQL and try again."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {"type": "string", "description": "A valid Spark SQL SELECT statement."}
                },
                "required": ["sql"],
            },
        },
    },
]

SYSTEM_PROMPT = f"""\
You are a database analyst assistant. You answer questions by querying a Spark SQL database.
The current catalog and schema are already set to `{CATALOG}.{SCHEMA}`, so use unqualified table names.

You have two tools:
  - get_schema: see the database tables and columns
  - run_query:  run a SELECT query (Spark SQL dialect)

Work through problems step by step:
  1. Think about what data you need (reason before acting)
  2. Call get_schema if you don't know the schema yet
  3. Write and run the SQL query
  4. If the query returns an error, read the error message and fix the SQL
  5. When you have enough data, summarise the answer clearly

Rules:
  - Only use SELECT statements \u2014 write operations are blocked
  - If a query fails, try once more with corrected SQL before giving up
  - Be concise in your final answer; lead with the key number or fact
"""

## Step 5 — The tool executor (errors as return values)

**Key teaching moment:** when a query fails, we **return** the error string instead of raising. That string becomes the next tool message the LLM sees — so it can read the error and retry with corrected SQL.

In [ ]:
def run_tool(name: str, arguments: dict) -> str:
    if name == "get_schema":
        return json.dumps(get_schema(), indent=2)

    if name == "run_query":
        sql = arguments.get("sql", "")
        validation = validate_sql(sql)
        if not validation.is_safe:
            return f"SAFETY_ERROR: {validation.reason}\nPlease fix the SQL and try again."
        try:
            columns, rows = run_query(sql)
            return json.dumps({"columns": columns, "rows": rows}, default=str)
        except Exception as exc:
            return f"DB_ERROR: {exc}\nPlease fix the SQL and try again."

    return f"UNKNOWN_TOOL: '{name}'"

## Step 6 — The agent loop

Pseudocode:

```
while steps < MAX_STEPS:
    1. Call LLM with current messages + tools
    2. If finish_reason == "stop"       → done, extract answer
    3. If finish_reason == "tool_calls" → execute tools, append results, continue
    4. If steps == MAX_STEPS            → give up, return partial answer
```

`history` is passed in so you can keep memory across multiple questions.

In [ ]:
@dataclass
class AgentResult:
    answer: str
    steps_taken: int
    tool_calls_made: list = field(default_factory=list)
    hit_step_limit: bool = False


def run_agent(question: str, history: list, verbose: bool = True) -> AgentResult:
    history.append({"role": "user", "content": question})
    tool_calls_log = []

    for step in range(1, MAX_STEPS + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=history,
            tools=TOOLS,
            temperature=0,
        )
        choice = response.choices[0]
        message = choice.message
        history.append(message)

        if verbose:
            print(f"  [step {step}] finish_reason={choice.finish_reason}")

        if choice.finish_reason == "stop":
            return AgentResult(answer=message.content or "", steps_taken=step, tool_calls_made=tool_calls_log)

        if choice.finish_reason == "tool_calls":
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                if verbose:
                    print(f"           \u2192 {fn_name}({json.dumps(fn_args) if fn_args else ''})")
                result = run_tool(fn_name, fn_args)
                tool_calls_log.append({"tool": fn_name, "args": fn_args, "result_preview": result[:80]})
                history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
                if verbose:
                    preview = result[:100].replace("\n", " ")
                    print(f"           \u2190 {preview}{'...' if len(result) > 100 else ''}")
        else:
            if verbose:
                print(f"  (unexpected finish_reason \u2014 stopping)")
            break

    return AgentResult(
        answer="Agent reached the maximum number of steps without completing.",
        steps_taken=MAX_STEPS,
        tool_calls_made=tool_calls_log,
        hit_step_limit=True,
    )

## Step 7 — Ask a single question

Fresh history each time = no memory across runs. The agent will discover the schema on its own.

In [ ]:
def ask_once(question: str):
    history = [{"role": "system", "content": SYSTEM_PROMPT}]
    print(f"\nQuestion: {question}")
    print("=" * 60)
    result = run_agent(question, history, verbose=True)
    print("\nAnswer:")
    print(result.answer)
    print("=" * 60)
    print(f"Steps: {result.steps_taken}/{MAX_STEPS}  |  Tool calls: {len(result.tool_calls_made)}")
    if result.hit_step_limit:
        print("\u26a0\ufe0f  Agent hit the step limit.")


ask_once("What tables are in this database, and roughly what do they contain?")

## Step 8 — Multi-turn memory

Share `history` across calls and the agent remembers earlier questions and answers. Notebook cells replace the terminal REPL from the original script — just call `chat(...)` multiple times.

In [ ]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(question: str):
    result = run_agent(question, history, verbose=False)
    print(f"Q: {question}")
    print(f"A: {result.answer}")
    print(f"[steps: {result.steps_taken} | history length: {len(history)} messages]\n")
    return result


chat("Pick the largest table in the database and tell me how many rows it has.");

In [ ]:
chat("Now show me the first 5 rows from that table.");

## Step 9 — Watch it recover from an error

This question references columns that almost certainly don't exist. The first SQL attempt will fail, the DB error gets fed back, and the agent should self-correct on the next step.

In [ ]:
ask_once("Find the row with the highest value in a column called grand_total — guess the table.")

## Summary

| Concept | Why it matters |
|---------|----------------|
| Returning errors as strings | Lets the LLM read the failure and self-correct |
| `MAX_STEPS` guard | Prevents an infinite loop when the model is stuck |
| Shared `history` | Gives the agent memory across questions |
| ReAct prompt | "Think, then act" — improves accuracy on multi-step questions |

**Try a different schema** by changing the `SCHEMA` widget at the top — `nyctaxi`, `accuweather`, `tpch`, etc. — and re-running. The agent introspects whatever you point it at.

**Next:** Module 05 — expose these same tools over the Model Context Protocol so any MCP-compatible client (Claude Desktop, Cursor, etc.) can use them.